# Traffic Analysis


In [ ]:
# importo librerie necessarie
import pandas as pd
import numpy as np
import requests
from io import StringIO

from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
#importo database Istat e controllo num righe e colonne
url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
headers = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}

response = requests.get(url, headers=headers)
print("Status:", response.status_code)

df = pd.read_csv(StringIO(response.text))
print("Righe e colonne:", df.shape)


Status: 200
Righe e colonne: (573552, 16)


In [ ]:
#controllo intestazioni colonne
df.columns.tolist()

['DATAFLOW',
 'FREQ',
 'REF_AREA',
 'DATA_TYPE',
 'RESULT',
 'TIME_PERIOD',
 'OBS_VALUE',
 'OBS_STATUS',
 'NOTE_DS',
 'NOTE_REF_AREA',
 'NOTE_DATA_TYPE',
 'NOTE_RESULT',
 'NOTE_TIME_PERIOD',
 'BASE_PER',
 'UNIT_MEAS',
 'UNIT_MULT']

In [ ]:
#anteprima database
df.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#verifico formati colonne
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 573552 entries, 0 to 573551
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   DATAFLOW          573552 non-null  object 
 1   FREQ              573552 non-null  object 
 2   REF_AREA          573552 non-null  int64  
 3   DATA_TYPE         573552 non-null  object 
 4   RESULT            573552 non-null  object 
 5   TIME_PERIOD       573552 non-null  int64  
 6   OBS_VALUE         573552 non-null  int64  
 7   OBS_STATUS        0 non-null       float64
 8   NOTE_DS           0 non-null       float64
 9   NOTE_REF_AREA     0 non-null       float64
 10  NOTE_DATA_TYPE    0 non-null       float64
 11  NOTE_RESULT       0 non-null       float64
 12  NOTE_TIME_PERIOD  0 non-null       float64
 13  BASE_PER          0 non-null       float64
 14  UNIT_MEAS         0 non-null       float64
 15  UNIT_MULT         0 non-null       float64
dtypes: float64(9), int64

In [ ]:
# analizzo colonna Data Type
df["DATA_TYPE"].value_counts()

,count
DATA_TYPE,
KILLINJ,382368
ROADACC,191184


In [ ]:
# anni di riferimento
df["TIME_PERIOD"].value_counts()

,count
TIME_PERIOD,
2002,24306
2003,24306
2001,24303
2005,24303
2007,24303
2006,24303
2008,24303
2004,24300
2009,24300


In [ ]:
# creo colonna incidenti e persone coinvolte e sistemo i tipi dato
incidenti = (
    df[df["DATA_TYPE"] == "ROADACC"]
    [["REF_AREA", "TIME_PERIOD", "OBS_VALUE"]]
    .rename(columns={
        "REF_AREA": "codice_territorio",
        "TIME_PERIOD": "anno",
        "OBS_VALUE": "incidenti"
    }))

coinvolti_grouped = (
    df[df["DATA_TYPE"] == "KILLINJ"]
    .groupby(["REF_AREA", "TIME_PERIOD"], as_index=False)["OBS_VALUE"]
    .sum()
    .rename(columns={
        "REF_AREA": "codice_territorio",
        "TIME_PERIOD": "anno",
        "OBS_VALUE": "persone_coinvolte"
    })
)

df_clean = incidenti.merge(
    coinvolti_grouped,
    on=["codice_territorio", "anno"],
    how="left"
)

df_clean["anno"] = df_clean["anno"].astype(int)
df_clean["incidenti"] = pd.to_numeric(df_clean["incidenti"], errors="coerce")
df_clean["persone_coinvolte"] = pd.to_numeric(df_clean["persone_coinvolte"], errors="coerce")

In [ ]:
# controllo colonne

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 191184 entries, 0 to 191183
Data columns (total 4 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   codice_territorio  191184 non-null  int64
 1   anno               191184 non-null  int64
 2   incidenti          191184 non-null  int64
 3   persone_coinvolte  191184 non-null  int64
dtypes: int64(4)
memory usage: 5.8 MB


In [ ]:
# verifico prime righe
df_clean.head()

,codice_territorio,anno,incidenti,persone_coinvolte
0,1001,2001,5,10
1,1001,2002,5,10
2,1001,2003,4,7
3,1001,2004,9,13
4,1001,2005,2,2


In [ ]:
# Carico il csv della popolazione
pop = pd.read_csv("Comuni - Dimensione Data Indagine 31-12-2020 Stampa 20052026215146.csv",sep=";")


In [ ]:
#guardo le prime righe
pop.head()

,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
0,1,1,1,201,1001,1001,Agliè,NaN,TO,2644,2011,"13,1462",2020,2545,2020
1,1,1,1,201,1002,1002,Airasca,NaN,TO,3819,2011,"15,7393",2020,3633,2020
2,1,1,1,201,1003,1003,Ala di Stura,NaN,TO,462,2011,"46,3315",2020,459,2020
3,1,1,1,201,1004,1004,Albiano d'Ivrea,NaN,TO,1791,2011,"11,7314",2020,1638,2020
4,1,1,1,201,1006,1006,Almese,NaN,TO,6303,2011,"17,8756",2020,6355,2020


In [ ]:
# guardo intestazioni colonne
pop.columns.tolist()


['Codice Ripartizione geografica',
 'Codice Regione',
 'Codice Provincia (Storico)',
 'Codice Provincia/Uts',
 'Codice Comune (alfanumerico)',
 'Codice Comune (numerico)',
 'Comune',
 'Comune (dizione straniera)',
 'Sigla automobilistica',
 'Popolazione legale',
 'Anno Censimento',
 'Superficie (Kmq)',
 'Anno (Superficie)',
 'Popolazione residente',
 'Anno (Popolazione residente)']

In [ ]:
# num righe e colonne

pop.shape

(7903, 15)

In [ ]:
#creo un dizionario per sostituire le sigle automobilistiche con i nomi completi delle province per la dashboard

province_map = {
    'TO': 'Torino', 'MI': 'Milano', 'RM': 'Roma', 'NA': 'Napoli', 'FI': 'Firenze',
    'BO': 'Bologna', 'GE': 'Genova', 'BA': 'Bari', 'PA': 'Palermo', 'CA': 'Cagliari',
    'TV': 'Treviso', 'PD': 'Padova', 'VE': 'Venezia', 'VR': 'Verona', 'VI': 'Vicenza',
    'TS': 'Trieste', 'UD': 'Udine', 'BG': 'Bergamo', 'BS': 'Brescia', 'CO': 'Como',
    'CR': 'Cremona', 'LC': 'Lecco', 'LO': 'Lodi', 'MN': 'Mantova', 'MB': 'Monza e della Brianza',
    'PV': 'Pavia', 'VA': 'Varese', 'AL': 'Alessandria', 'AT': 'Asti', 'CN': 'Cuneo',
    'NO': 'Novara', 'VC': 'Vercelli', 'AO': 'Aosta', 'IM': 'Imperia', 'SV': 'Savona',
    'SP': 'La Spezia', 'MS': 'Massa-Carrara', 'LU': 'Lucca', 'PI': 'Pisa', 'PT': 'Pistoia',
    'PO': 'Prato', 'SI': 'Siena', 'AR': 'Arezzo', 'GR': 'Grosseto', 'LI': 'Livorno',
    'PG': 'Perugia', 'TR': 'Terni', 'AN': 'Ancona', 'AP': 'Ascoli Piceno', 'FM': 'Fermo',
    'MC': 'Macerata', 'PS': 'Pesaro e Urbino', 'CH': 'Chieti', 'AQ': 'L\'Aquila', 'PE': 'Pescara',
    'TE': 'Teramo', 'CB': 'Campobasso', 'IS': 'Isernia', 'FR': 'Frosinone', 'LT': 'Latina',
    'RI': 'Rieti', 'VT': 'Viterbo', 'CE': 'Caserta', 'AV': 'Avellino', 'BN': 'Benevento',
    'SA': 'Salerno', 'FG': 'Foggia', 'LE': 'Lecce', 'TA': 'Taranto', 'BR': 'Brindisi',
    'CS': 'Cosenza', 'CZ': 'Catanzaro', 'KR': 'Crotone', 'RC': 'Reggio Calabria',
    'VV': 'Vibo Valentia', 'AG': 'Agrigento', 'CL': 'Caltanissetta', 'CT': 'Catania',
    'EN': 'Enna', 'ME': 'Messina', 'RG': 'Ragusa', 'SR': 'Siracusa', 'TP': 'Trapani',
    'NU': 'Nuoro', 'OR': 'Oristano', 'SS': 'Sassari', 'SU': 'Sud Sardegna', 'BZ': 'Bolzano',
    'TN': 'Trento', 'BL': 'Belluno', 'GO': 'Gorizia', 'PN': 'Pordenone', 'RO': 'Rovigo',
    'SO': 'Sondrio', 'UD': 'Udine', 'RN': 'Rimini', 'FC': 'Forlì-Cesena',
    'PR': 'Parma', 'PC': 'Piacenza', 'RA': 'Ravenna', 'RE': 'Reggio Emilia', 'MO': 'Modena',
    'BT': 'Barletta-Andria-Trani', 'MT': 'Matera', 'PZ': 'Potenza',
    'PU': 'Pesaro e Urbino','BI': 'Biella', 'FE': 'Ferrara', 'VB': 'Verbano-Cusio-Ossola'}


In [ ]:
# sistemo colonne
territori = pop[[
    "Codice Comune (numerico)",
    "Comune",
    "Sigla automobilistica",
    "Codice Regione",
    "Popolazione residente"]].copy()

territori.columns = [
    "codice_territorio",
    "comune",
    "provincia",
    "regione",
    "popolazione"]

territori["codice_territorio"] = pd.to_numeric(territori["codice_territorio"], errors="coerce")
territori["provincia"] = territori["provincia"].map(province_map).fillna(territori["provincia"])

territori.head()


,codice_territorio,comune,provincia,regione,popolazione
0,1001,Agliè,Torino,1,2545
1,1002,Airasca,Torino,1,3633
2,1003,Ala di Stura,Torino,1,459
3,1004,Albiano d'Ivrea,Torino,1,1638
4,1006,Almese,Torino,1,6355


### Controllo tabella territori

Verifico che i nomi delle province siano leggibili e che codice territorio e popolazione siano presenti.

In [ ]:
# vedo se ci sono N/A
territori.isna().sum()

,0
codice_territorio,0
comune,1
provincia,92
regione,0
popolazione,0


In [ ]:
#creo database per dashboard facendo il merge tra i due database di input
df_finale = df_clean.merge(
    territori,
    on="codice_territorio",
    how="left"
)


In [ ]:
#elimino riga in cui il comune è N/A e sostituisco le province N/A con "sconosciuta"
df_finale = df_finale.dropna(subset=["comune", "popolazione"])
df_finale["provincia"] = df_finale["provincia"].fillna("Sconosciuta")

In [ ]:
#creo indicatori incidenti e persone coinvolte ogni 1000 abitanti
df_finale["incidenti_per_1000"] = (df_finale["incidenti"] / df_finale["popolazione"]) * 1000
df_finale["coinvolti_per_1000"] = (df_finale["persone_coinvolte"] / df_finale["popolazione"]) * 1000

In [ ]:
#elimino comuni con popolazione inferiore a 5k
df_finale = df_finale[df_finale["popolazione"] >= 5000]

In [ ]:
#elimino outliers
Q1 = df_finale["incidenti_per_1000"].quantile(0.25)
Q3 = df_finale["incidenti_per_1000"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_finale = df_finale[
    (df_finale["incidenti_per_1000"] >= lower_bound) &
    (df_finale["incidenti_per_1000"] <= upper_bound)
]

In [ ]:
#controllo prime righe
df_finale.head()

,codice_territorio,anno,incidenti,persone_coinvolte,comune,provincia,regione,popolazione,incidenti_per_1000,coinvolti_per_1000
114,1006,2001,7,11,Almese,Torino,1.0,6355.0,1.101495,1.730921
115,1006,2002,5,6,Almese,Torino,1.0,6355.0,0.786782,0.944138
116,1006,2003,6,13,Almese,Torino,1.0,6355.0,0.944138,2.045633
117,1006,2004,1,1,Almese,Torino,1.0,6355.0,0.157356,0.157356
118,1006,2005,8,11,Almese,Torino,1.0,6355.0,1.258851,1.730921


In [ ]:
df_finale.columns.tolist()

['codice_territorio',
 'anno',
 'incidenti',
 'persone_coinvolte',
 'comune',
 'provincia',
 'regione',
 'popolazione',
 'incidenti_per_1000',
 'coinvolti_per_1000']

In [ ]:
#controllo se ci sono N/A
df_finale.isna().sum()

,0
codice_territorio,0
anno,0
incidenti,0
persone_coinvolte,0
comune,0
provincia,0
regione,0
popolazione,0
incidenti_per_1000,0
coinvolti_per_1000,0


In [ ]:
#analisi valori
df_finale.describe()

,codice_territorio,anno,incidenti,persone_coinvolte,regione,popolazione,incidenti_per_1000,coinvolti_per_1000
count,53200.000000,53200.000000,53200.000000,53200.000000,53200.000000,5.320000e+04,53200.000000,53200.000000
mean,47366.414906,2012.869850,57.345620,82.545921,9.419098,1.893390e+04,2.351225,3.508709
std,28851.181900,6.876409,267.492066,356.466692,5.926697,5.940197e+04,1.362321,2.041838
min,1006.000000,2001.000000,0.000000,0.000000,1.000000,5.002000e+03,0.000000,0.000000
25%,21086.000000,2007.000000,11.000000,17.000000,3.000000,6.887000e+03,1.332889,1.979078
50%,47002.000000,2013.000000,22.000000,33.000000,8.000000,1.005300e+04,2.163566,3.231369
75%,72016.000000,2019.000000,46.000000,68.000000,15.000000,1.736700e+04,3.187645,4.766361
max,111106.000000,2024.000000,15783.000000,20825.000000,20.000000,2.770226e+06,6.316031,14.749725


In [ ]:
#faccio regressione lineare su anno 2025 e 2026. escludo il 2020 perché ha valori bassi dovuti alla pandemia e mi altera le statistiche

forecast_results = []

for comune in df_finale["comune"].unique():
    temp = df_finale[(df_finale["comune"] == comune) & (df_finale["anno"] != 2020)]

    if len(temp) < 2:
        forecast_results.append({
            "comune": comune,
            "forecast_2025": np.nan,
            "forecast_2026": np.nan
        })
        continue

    model = LinearRegression()
    model.fit(temp[["anno"]], temp["incidenti"])

    future_years = pd.DataFrame({"anno": [2025, 2026]})
    predictions = model.predict(future_years)

    forecast_results.append({
        "comune": comune,
        "forecast_2025": predictions[0],
        "forecast_2026": predictions[1]
    })

forecast_df = pd.DataFrame(forecast_results)

df_finale = df_finale.merge(forecast_df, on="comune", how="left")


In [ ]:
#controllo prime righe
df_finale.head()

,codice_territorio,anno,incidenti,persone_coinvolte,comune,provincia,regione,popolazione,incidenti_per_1000,coinvolti_per_1000,forecast_2025,forecast_2026
0,1006,2001,7,11,Almese,Torino,1.0,6355.0,1.101495,1.730921,3.96753,3.866693
1,1006,2002,5,6,Almese,Torino,1.0,6355.0,0.786782,0.944138,3.96753,3.866693
2,1006,2003,6,13,Almese,Torino,1.0,6355.0,0.944138,2.045633,3.96753,3.866693
3,1006,2004,1,1,Almese,Torino,1.0,6355.0,0.157356,0.157356,3.96753,3.866693
4,1006,2005,8,11,Almese,Torino,1.0,6355.0,1.258851,1.730921,3.96753,3.866693


In [ ]:
#elimino colonne _x e _y
if 'forecast_2025_x' in df_finale.columns and 'forecast_2025_y' in df_finale.columns:
    df_finale['forecast_2025'] = df_finale['forecast_2025_y']
    df_finale = df_finale.drop(columns=['forecast_2025_x', 'forecast_2025_y'])

if 'forecast_2026_x' in df_finale.columns and 'forecast_2026_y' in df_finale.columns:
    df_finale['forecast_2026'] = df_finale['forecast_2026_y']
    df_finale = df_finale.drop(columns=['forecast_2026_x', 'forecast_2026_y'])

In [ ]:
#controllo che sia andato a buon fine
df_finale.head()

,codice_territorio,anno,incidenti,persone_coinvolte,comune,provincia,regione,popolazione,incidenti_per_1000,coinvolti_per_1000,forecast_2025,forecast_2026
0,1006,2001,7,11,Almese,Torino,1.0,6355.0,1.101495,1.730921,3.96753,3.866693
1,1006,2002,5,6,Almese,Torino,1.0,6355.0,0.786782,0.944138,3.96753,3.866693
2,1006,2003,6,13,Almese,Torino,1.0,6355.0,0.944138,2.045633,3.96753,3.866693
3,1006,2004,1,1,Almese,Torino,1.0,6355.0,0.157356,0.157356,3.96753,3.866693
4,1006,2005,8,11,Almese,Torino,1.0,6355.0,1.258851,1.730921,3.96753,3.866693


In [ ]:
#vedo se ho N/A nelle colonne che ho creato
display(df_finale[df_finale['forecast_2025'].isna()])
display(df_finale[df_finale['forecast_2026'].isna()])

,codice_territorio,anno,incidenti,persone_coinvolte,comune,provincia,regione,popolazione,incidenti_per_1000,coinvolti_per_1000,forecast_2025,forecast_2026
3400,10025,2020,2840,3351,Genova,Genova,7.0,566410.0,5.014036,5.916209,NaN,NaN
5882,15011,2020,33,46,Assago,Milano,3.0,9260.0,3.563715,4.967603,NaN,NaN
5883,15011,2021,36,46,Assago,Milano,3.0,9260.0,3.887689,4.967603,NaN,NaN
8222,16024,2014,681,929,Bergamo,Bergamo,3.0,119993.0,5.675331,7.742118,NaN,NaN
8223,16024,2020,620,775,Bergamo,Bergamo,3.0,119993.0,5.166968,6.458710,NaN,NaN
18070,27019,2020,131,163,Jesolo,Venezia,5.0,26511.0,4.941345,6.148391,NaN,NaN
18071,27019,2022,152,219,Jesolo,Venezia,5.0,26511.0,5.733469,8.260722,NaN,NaN
23258,37005,2020,30,42,Bentivoglio,Bologna,8.0,5801.0,5.171522,7.240131,NaN,NaN
51370,99013,2020,161,192,Riccione,Rimini,8.0,34942.0,4.607636,5.494820,NaN,NaN
51371,99013,2021,206,245,Riccione,Rimini,8.0,34942.0,5.895484,7.011619,NaN,NaN


,codice_territorio,anno,incidenti,persone_coinvolte,comune,provincia,regione,popolazione,incidenti_per_1000,coinvolti_per_1000,forecast_2025,forecast_2026
3400,10025,2020,2840,3351,Genova,Genova,7.0,566410.0,5.014036,5.916209,NaN,NaN
5882,15011,2020,33,46,Assago,Milano,3.0,9260.0,3.563715,4.967603,NaN,NaN
5883,15011,2021,36,46,Assago,Milano,3.0,9260.0,3.887689,4.967603,NaN,NaN
8222,16024,2014,681,929,Bergamo,Bergamo,3.0,119993.0,5.675331,7.742118,NaN,NaN
8223,16024,2020,620,775,Bergamo,Bergamo,3.0,119993.0,5.166968,6.458710,NaN,NaN
18070,27019,2020,131,163,Jesolo,Venezia,5.0,26511.0,4.941345,6.148391,NaN,NaN
18071,27019,2022,152,219,Jesolo,Venezia,5.0,26511.0,5.733469,8.260722,NaN,NaN
23258,37005,2020,30,42,Bentivoglio,Bologna,8.0,5801.0,5.171522,7.240131,NaN,NaN
51370,99013,2020,161,192,Riccione,Rimini,8.0,34942.0,4.607636,5.494820,NaN,NaN
51371,99013,2021,206,245,Riccione,Rimini,8.0,34942.0,5.895484,7.011619,NaN,NaN


In [ ]:
#creo dei cluster di rischio per ogni comune
cluster_cols = [
    "incidenti_per_1000",
    "coinvolti_per_1000",
    "forecast_2025",
    "forecast_2026"
]

cluster_df = (
    df_finale.groupby("comune", as_index=False)[cluster_cols]
    .mean()
)

cluster_df_for_kmeans = cluster_df.dropna(subset=cluster_cols).copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df_for_kmeans[cluster_cols])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_df_for_kmeans["cluster"] = kmeans.fit_predict(X_scaled)

cluster_df = cluster_df.merge(
    cluster_df_for_kmeans[["comune", "cluster"]],
    on="comune",
    how="left"
)

cluster_means = cluster_df.groupby("cluster")["incidenti_per_1000"].mean().sort_values(ascending=False)

cluster_labels = {
    cluster_means.index[0]: "Alto Rischio",
    cluster_means.index[1]: "Medio Rischio",
    cluster_means.index[2]: "Basso Rischio"
}

cluster_df["livello_rischio"] = cluster_df["cluster"].map(cluster_labels)

cluster_df.head()


,comune,incidenti_per_1000,coinvolti_per_1000,forecast_2025,forecast_2026,cluster,livello_rischio
0,Abano Terme,3.166211,3.997678,47.494821,46.119841,1.0,Medio Rischio
1,Abbadia San Salvatore,1.892392,2.811360,2.177689,1.442390,0.0,Basso Rischio
2,Abbiategrasso,3.775865,4.853584,73.024701,69.013068,1.0,Medio Rischio
3,Acate,0.969083,1.910703,7.635060,7.413386,0.0,Basso Rischio
4,Acerra,1.189272,1.872121,80.888845,81.741195,0.0,Basso Rischio


In [ ]:
#vedo i valori della colonna appena creata
print(cluster_df["livello_rischio"].value_counts(dropna=False))

livello_rischio
Basso Rischio    1392
Medio Rischio     969
NaN                 6
Alto Rischio        2
Name: count, dtype: int64


In [ ]:
#guardo N/A dei cluster
display(cluster_df[cluster_df['livello_rischio'].isna()])

,comune,incidenti_per_1000,coinvolti_per_1000,forecast_2025,forecast_2026,cluster,livello_rischio
124,Assago,3.725702,4.967603,NaN,NaN,NaN,NaN
194,Bentivoglio,5.171522,7.240131,NaN,NaN,NaN,NaN
195,Bergamo,5.421150,7.100414,NaN,NaN,NaN,NaN
924,Genova,5.014036,5.916209,NaN,NaN,NaN,NaN
1018,Jesolo,5.337407,7.204557,NaN,NaN,NaN,NaN
1696,Riccione,5.251560,6.253220,NaN,NaN,NaN,NaN


In [ ]:
#creo database per dashboard selezionando le colonne
df_finale = df_finale.merge(
    cluster_df[["comune", "cluster", "livello_rischio"]],
    on="comune",
    how="left"
)

df_finale = df_finale.drop_duplicates()

colonne_finali = [
    "codice_territorio",
    "anno",
    "incidenti",
    "persone_coinvolte",
    "comune",
    "provincia",
    "regione",
    "popolazione",
    "incidenti_per_1000",
    "coinvolti_per_1000",
    "forecast_2025",
    "forecast_2026",
    "cluster",
    "livello_rischio"
]

df_finale = df_finale[colonne_finali]


df_finale.head()


,codice_territorio,anno,incidenti,persone_coinvolte,comune,provincia,regione,popolazione,incidenti_per_1000,coinvolti_per_1000,forecast_2025,forecast_2026,cluster,livello_rischio
0,1006,2001,7,11,Almese,Torino,1.0,6355.0,1.101495,1.730921,3.96753,3.866693,0.0,Basso Rischio
1,1006,2002,5,6,Almese,Torino,1.0,6355.0,0.786782,0.944138,3.96753,3.866693,0.0,Basso Rischio
2,1006,2003,6,13,Almese,Torino,1.0,6355.0,0.944138,2.045633,3.96753,3.866693,0.0,Basso Rischio
3,1006,2004,1,1,Almese,Torino,1.0,6355.0,0.157356,0.157356,3.96753,3.866693,0.0,Basso Rischio
4,1006,2005,8,11,Almese,Torino,1.0,6355.0,1.258851,1.730921,3.96753,3.866693,0.0,Basso Rischio


In [ ]:
#esporto CSV
df_finale.to_csv(
    "traffic_accidents_final.csv",
    index=False,
    sep=";",
    decimal=","
)

print("File creato: traffic_accidents_final.csv")


File creato: traffic_accidents_final.csv


In [ ]:
#scarico file
from google.colab import files
files.download("traffic_accidents_final.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>